# V2G Fleet AutoResearch — Analysis

Explore the Nord Pool DK1 price data, EV fleet session patterns,
and autonomous experiment results from `results_v2g.tsv`.

In [ ]:
import json
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

CACHE_DIR = Path.home() / '.cache' / 'autoresearch_v2g'
print('Cache dir:', CACHE_DIR)
print('Files:', [f.name for f in CACHE_DIR.iterdir()] if CACHE_DIR.exists() else 'NOT FOUND — run: uv run prepare_v2g.py')

: 

In [ ]:
# Load all cached data
prices = np.load(CACHE_DIR / 'prices_eur_mwh.npy')          # (N_days, 24)
with open(CACHE_DIR / 'sessions.json') as f:
    sessions = json.load(f)
with open(CACHE_DIR / 'dates.json') as f:
    dates = [datetime.date.fromisoformat(s) for s in json.load(f)]

TRAIN_END  = datetime.date(2022, 12, 31)
VAL_START  = datetime.date(2023, 1, 1)
VAL_END    = datetime.date(2023, 12, 31)
TEST_START = datetime.date(2024, 1, 1)

print(f'Loaded {len(dates)} days  |  prices shape: {prices.shape}')
print(f'Price range: {prices.min():.1f} – {prices.max():.1f} EUR/MWh')
print(f'Train: {dates[0]} – {TRAIN_END}')
print(f'Val:   {VAL_START} – {VAL_END}')
print(f'Test:  {TEST_START} – {dates[-1]}')

---
## 1. Electricity Prices (Nord Pool DK1)

Day-ahead spot prices in EUR/MWh. The price spread across hours each day is what V2G arbitrage exploits.

In [ ]:
# Daily stats: mean, min, max, spread
daily_mean  = prices.mean(axis=1)
daily_min   = prices.min(axis=1)
daily_max   = prices.max(axis=1)
daily_spread = daily_max - daily_min
date_arr = np.array(dates)

fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

# Background shading for splits
split_colors = {'train': '#e8f5e9', 'val': '#e3f2fd', 'test': '#fff3e0'}
for ax in axes:
    ax.axvspan(dates[0], TRAIN_END, alpha=0.3, color=split_colors['train'], label='Train')
    ax.axvspan(VAL_START, VAL_END, alpha=0.3, color=split_colors['val'], label='Val (agent optimizes)')
    ax.axvspan(TEST_START, dates[-1], alpha=0.3, color=split_colors['test'], label='Test (held out)')

# Panel 1: Daily mean price
axes[0].plot(dates, daily_mean, lw=0.8, color='#1565c0', alpha=0.8)
axes[0].fill_between(dates, daily_min, daily_max, alpha=0.15, color='#1565c0')
axes[0].axhline(0, color='red', lw=0.8, ls='--', alpha=0.5)
axes[0].set_ylabel('Price (EUR/MWh)')
axes[0].set_title('Nord Pool DK1 Day-Ahead Prices', fontsize=13)
axes[0].legend(loc='upper right', fontsize=8, ncol=3)

# Panel 2: Daily price spread (V2G opportunity)
axes[1].fill_between(dates, 0, daily_spread, alpha=0.6, color='#e65100')
axes[1].set_ylabel('Daily Spread (EUR/MWh)')
axes[1].set_title('Daily Price Spread — V2G Arbitrage Opportunity', fontsize=11)
axes[1].axhline(100, color='gray', lw=0.8, ls='--', alpha=0.5, label='>100 EUR/MWh spread')
axes[1].legend(fontsize=8)

# Panel 3: Negative price hours per day
neg_hours = (prices < 0).sum(axis=1)
axes[2].bar(dates, neg_hours, color='#6a1b9a', alpha=0.7, width=1)
axes[2].set_ylabel('Hours with negative price')
axes[2].set_xlabel('Date')
axes[2].set_title('Negative Price Hours per Day — Free/Paid Charging Opportunities', fontsize=11)
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('prices_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Days with ≥1 negative price hour: {(neg_hours > 0).sum()} / {len(dates)}')
print(f'Days with spread > 100 EUR/MWh:   {(daily_spread > 100).sum()} / {len(dates)}')

In [ ]:
# Average hourly price profile by season
date_arr = np.array(dates)
months = np.array([d.month for d in dates])
seasons = {'Winter (Dec-Feb)': [12,1,2], 'Spring (Mar-May)': [3,4,5],
           'Summer (Jun-Aug)': [6,7,8], 'Autumn (Sep-Nov)': [9,10,11]}
colors  = ['#1565c0', '#2e7d32', '#f57f17', '#b71c1c']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
for (season, mnths), color in zip(seasons.items(), colors):
    mask = np.isin(months, mnths)
    profile = prices[mask].mean(axis=0)
    ax.plot(range(24), profile, label=season, color=color, lw=2)
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('Hour of day (UTC)')
ax.set_ylabel('Avg Price (EUR/MWh)')
ax.set_title('Average Hourly Price Profile by Season')
ax.set_xticks(range(0, 24, 2))
ax.legend(fontsize=9)

# Price distribution histogram
ax2 = axes[1]
flat = prices.flatten()
ax2.hist(flat[flat < 300], bins=80, color='#1565c0', alpha=0.7, edgecolor='none')
ax2.axvline(0, color='red', lw=1.5, ls='--', label='Zero price')
ax2.axvline(50, color='orange', lw=1.5, ls='--', label='50 EUR/MWh (degrad. cost equiv.)')
ax2.set_xlabel('Price (EUR/MWh)')
ax2.set_ylabel('Frequency')
ax2.set_title('Price Distribution (all hours, all years)')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()
print(f'Hours below 0 EUR/MWh: {(flat < 0).sum():,} ({(flat < 0).mean()*100:.1f}%)')
print(f'Hours above 100 EUR/MWh: {(flat > 100).sum():,} ({(flat > 100).mean()*100:.1f}%)')

---
## 2. EV Fleet Session Patterns

30-vehicle workplace depot (15 V2G capable). Sessions generated from realistic ACN-Data statistics.

In [ ]:
# Flatten all sessions into a DataFrame
rows = []
for i, (day_sessions, date) in enumerate(zip(sessions, dates)):
    for sess in day_sessions:
        rows.append({
            'date': date,
            'weekday': date.weekday(),
            'is_weekend': date.weekday() >= 5,
            'vehicle_id': sess['vehicle_id'],
            'arrival_hour': sess['arrival_hour'],
            'departure_hour': sess['departure_hour'],
            'duration_h': sess['departure_hour'] - sess['arrival_hour'],
            'energy_needed_kwh': sess['energy_needed_kwh'],
            'soc_arrival': sess['soc_arrival'],
            'v2g_capable': sess['max_discharge_kw'] > 0,
        })
df_sess = pd.DataFrame(rows)
print(f'Total sessions: {len(df_sess):,}  |  {len(dates)} days')
print(f'V2G capable: {df_sess.v2g_capable.sum():,} sessions ({df_sess.v2g_capable.mean()*100:.0f}%)')
print(f'Avg vehicles/weekday: {df_sess[~df_sess.is_weekend].groupby("date").size().mean():.1f}')
print(f'Avg vehicles/weekend: {df_sess[df_sess.is_weekend].groupby("date").size().mean():.1f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))

# Arrival distribution
ax = axes[0, 0]
for label, mask, color in [('Weekday', ~df_sess.is_weekend, '#1565c0'),
                             ('Weekend', df_sess.is_weekend, '#e65100')]:
    ax.hist(df_sess[mask]['arrival_hour'], bins=range(24), alpha=0.6,
            color=color, label=label, density=True, rwidth=0.8)
ax.set_xlabel('Arrival Hour')
ax.set_ylabel('Density')
ax.set_title('Vehicle Arrival Distribution')
ax.legend()

# Departure distribution
ax = axes[0, 1]
for label, mask, color in [('Weekday', ~df_sess.is_weekend, '#1565c0'),
                             ('Weekend', df_sess.is_weekend, '#e65100')]:
    ax.hist(df_sess[mask]['departure_hour'], bins=range(24), alpha=0.6,
            color=color, label=label, density=True, rwidth=0.8)
ax.set_xlabel('Departure Hour')
ax.set_title('Vehicle Departure Distribution')
ax.legend()

# Session duration
ax = axes[0, 2]
ax.hist(df_sess['duration_h'], bins=range(1, 18), color='#2e7d32', alpha=0.7, rwidth=0.8, density=True)
ax.set_xlabel('Session Duration (hours)')
ax.set_title('Session Duration Distribution')

# Energy needed
ax = axes[1, 0]
ax.hist(df_sess['energy_needed_kwh'], bins=30, color='#6a1b9a', alpha=0.7)
ax.set_xlabel('Energy Needed (kWh)')
ax.set_title('Energy Requested per Session')
ax.axvline(df_sess['energy_needed_kwh'].mean(), color='red', lw=2, ls='--',
           label=f"Mean: {df_sess['energy_needed_kwh'].mean():.1f} kWh")
ax.legend()

# SoC on arrival
ax = axes[1, 1]
ax.hist(df_sess['soc_arrival'], bins=30, color='#f57f17', alpha=0.7)
ax.set_xlabel('SoC on Arrival')
ax.set_title('Battery SoC at Arrival')
ax.axvline(0.80, color='red', lw=2, ls='--', label='80% departure target')
ax.legend()

# Fleet occupancy by hour (avg vehicles plugged in)
ax = axes[1, 2]
hourly_occ = np.zeros(24)
for _, sess in df_sess.iterrows():
    for h in range(int(sess['arrival_hour']), int(sess['departure_hour'])):
        hourly_occ[h] += 1
hourly_occ /= len(dates)
v2g_occ = np.zeros(24)
for _, sess in df_sess[df_sess.v2g_capable].iterrows():
    for h in range(int(sess['arrival_hour']), int(sess['departure_hour'])):
        v2g_occ[h] += 1
v2g_occ /= len(dates)
ax.bar(range(24), hourly_occ, color='#1565c0', alpha=0.5, label='All vehicles')
ax.bar(range(24), v2g_occ, color='#e65100', alpha=0.7, label='V2G capable')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Avg Vehicles Plugged In')
ax.set_title('Fleet Occupancy by Hour')
ax.legend()

plt.suptitle('EV Fleet Session Patterns (30-vehicle workplace depot)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('fleet_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Experiment Results

Load `results_v2g.tsv` and visualize how `val_revenue_per_kwh` improves as the agent iterates.

In [ ]:
try:
    df = pd.read_csv('results_v2g.tsv', sep='\t')
    df['val_revenue_per_kwh'] = pd.to_numeric(df['val_revenue_per_kwh'], errors='coerce')
    df['violations'] = pd.to_numeric(df['violations'], errors='coerce').fillna(0)
    df['status'] = df['status'].str.strip().str.upper()
    print(f'Loaded {len(df)} experiments')
    print(df['status'].value_counts().to_string())
    print()
    kept = df[df['status'] == 'KEEP']
    print(f'Baseline:  {df.iloc[0]["val_revenue_per_kwh"]:.6f} EUR/kWh')
    if len(kept) > 1:
        print(f'Best:      {kept["val_revenue_per_kwh"].max():.6f} EUR/kWh')
        print(f'Improvement: {kept["val_revenue_per_kwh"].max() - df.iloc[0]["val_revenue_per_kwh"]:+.6f} EUR/kWh')
    df.head(10)
except FileNotFoundError:
    print('results_v2g.tsv not found — start the agent loop first.')
    df = None

In [ ]:
if df is not None and len(df) > 1:
    fig, axes = plt.subplots(2, 1, figsize=(16, 10))

    valid = df[df['status'] != 'CRASH'].reset_index(drop=True)
    baseline_rev = valid.loc[0, 'val_revenue_per_kwh']

    # Panel 1: Revenue over experiments
    ax = axes[0]
    disc = valid[valid['status'] == 'DISCARD']
    kept_v = valid[valid['status'] == 'KEEP']

    ax.scatter(disc.index, disc['val_revenue_per_kwh'],
               c='#cccccc', s=18, alpha=0.6, zorder=2, label='Discarded')
    ax.scatter(kept_v.index, kept_v['val_revenue_per_kwh'],
               c='#2ecc71', s=60, zorder=4, edgecolors='black',
               linewidths=0.5, label='Kept')

    # Running best line (cumulative max of kept experiments)
    kept_mask = valid['status'] == 'KEEP'
    kept_idx = valid.index[kept_mask]
    kept_rev = valid.loc[kept_mask, 'val_revenue_per_kwh']
    running_best = kept_rev.cummax()
    ax.step(kept_idx, running_best, where='post', color='#27ae60',
            lw=2.5, alpha=0.9, zorder=3, label='Running best')

    ax.axhline(0, color='red', lw=1.5, ls='--', alpha=0.6, label='Break-even (0)')
    ax.axhline(baseline_rev, color='gray', lw=1, ls=':', alpha=0.7, label=f'Baseline ({baseline_rev:.4f})')

    # Label kept experiments
    for idx, rev in zip(kept_idx, kept_rev):
        desc = str(valid.loc[idx, 'description'])[:45]
        ax.annotate(desc, (idx, rev), textcoords='offset points',
                    xytext=(6, 4), fontsize=7.5, color='#1a7a3a',
                    rotation=25, ha='left', va='bottom')

    n_kept = len(kept_v)
    n_total = len(df)
    ax.set_ylabel('val_revenue_per_kwh (EUR/kWh)')
    ax.set_title(f'V2G AutoResearch Progress: {n_total} experiments, {n_kept} kept improvements',
                 fontsize=13)
    ax.legend(loc='lower right', fontsize=9)

    # Panel 2: Constraint violations over experiments
    ax2 = axes[1]
    colors_bar = {'KEEP': '#2ecc71', 'DISCARD': '#e74c3c', 'CRASH': '#95a5a6'}
    bar_colors = [colors_bar.get(s, '#aaaaaa') for s in df['status']]
    ax2.bar(range(len(df)), df['violations'], color=bar_colors, alpha=0.75, width=0.8)
    ax2.set_xlabel('Experiment #')
    ax2.set_ylabel('Constraint Violations')
    ax2.set_title('Departure SoC Violations per Experiment (green=kept, red=discarded)')

    plt.tight_layout()
    plt.savefig('progress.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved to progress.png')
else:
    print('Not enough experiments to plot. Run the agent loop first.')

In [ ]:
# Summary table of all kept experiments
if df is not None:
    kept = df[df['status'] == 'KEEP'].copy()
    if len(kept) > 1:
        kept['delta'] = kept['val_revenue_per_kwh'].diff()
        hits = kept.iloc[1:].sort_values('delta', ascending=False)
        print(f'{"Rank":>4}  {"Delta":>10}  {"Revenue/kWh":>12}  {"Violations":>10}  Description')
        print('-' * 85)
        for rank, (_, row) in enumerate(hits.iterrows(), 1):
            print(f'{rank:4d}  {row["delta"]:+.6f}  {row["val_revenue_per_kwh"]:12.6f}  '
                  f'{int(row["violations"]):10d}  {row["description"]}')
        print()
        total = kept['val_revenue_per_kwh'].iloc[-1] - kept['val_revenue_per_kwh'].iloc[0]
        print(f'Total improvement over baseline: {total:+.6f} EUR/kWh')

---
## 4. Strategy Deep Dive — One Day's Dispatch

Visualize what the current strategy does on a single example day:
actual prices vs the charge/discharge schedule chosen.

In [ ]:
import importlib.util, sys
sys.path.insert(0, '.')

# Load strategy and run on one val day
spec = importlib.util.spec_from_file_location('strategy', 'strategy.py')
strategy = importlib.util.module_from_spec(spec)
spec.loader.exec_module(strategy)

# Build train split for fitting
train_end_date = datetime.date(2022, 12, 31)
train_idx = [i for i, d in enumerate(dates) if d <= train_end_date]
strategy.fit(prices[train_idx], [sessions[i] for i in train_idx])

# Pick an interesting example day (high spread day in val period)
val_mask = np.array([datetime.date(2023,1,1) <= d <= datetime.date(2023,12,31) for d in dates])
val_indices = np.where(val_mask)[0]
val_spreads = (prices[val_mask].max(axis=1) - prices[val_mask].min(axis=1))
best_val_idx = val_indices[val_spreads.argmax()]   # day with highest price spread
example_date = dates[best_val_idx]
print(f'Example day: {example_date}  (price spread: {val_spreads.max():.1f} EUR/MWh)')

In [ ]:
# Get strategy plan for that day
hist_start = max(0, best_val_idx - 30)
price_history   = prices[hist_start:best_val_idx]
session_history = sessions[hist_start:best_val_idx]
today_sessions  = sessions[best_val_idx]
actual_prices   = prices[best_val_idx]

schedule = strategy.plan_day(
    date=example_date,
    price_history=price_history,
    session_history=session_history,
    vehicle_states=today_sessions,
)
schedule = np.array(schedule)
print(f'Schedule shape: {schedule.shape}  |  {len(today_sessions)} vehicles today')

# Aggregate fleet-level power
fleet_charge    = np.maximum(schedule, 0).sum(axis=0)   # total charging kW
fleet_discharge = np.minimum(schedule, 0).sum(axis=0)   # total discharging kW
fleet_net       = fleet_charge + fleet_discharge         # net grid draw

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
hours = np.arange(24)

# Panel 1: Actual prices vs forecast
ax = axes[0]
ax.bar(hours, actual_prices, color=[
    '#e53935' if p > 80 else '#43a047' if p < 20 else '#90a4ae'
    for p in actual_prices
], alpha=0.7, width=0.8, label='Actual price')
ax.axhline(0, color='black', lw=0.8)
if hasattr(strategy, 'last_price_forecast') and strategy.last_price_forecast is not None:
    fc = np.array(strategy.last_price_forecast)
    ax.step(hours, fc, where='mid', color='navy', lw=2, ls='--', label=f'Forecast (RMSE={np.sqrt(np.mean((fc - actual_prices)**2)):.1f})')
ax.set_ylabel('Price (EUR/MWh)')
ax.set_title(f'Day: {example_date} — Actual Prices vs Strategy Forecast', fontsize=12)
ax.legend(fontsize=9)

# Panel 2: Fleet dispatch schedule
ax = axes[1]
ax.bar(hours - 0.2, fleet_charge, width=0.4, color='#1565c0', alpha=0.8, label='Charging (kW)')
ax.bar(hours + 0.2, -fleet_discharge, width=0.4, color='#e65100', alpha=0.8, label='V2G Discharging (kW)')
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('Fleet Power (kW)')
ax.set_title('Fleet Charge/Discharge Schedule', fontsize=12)
ax.legend(fontsize=9)

# Panel 3: Per-vehicle SoC evolution
ax = axes[2]
for i, sess in enumerate(today_sessions[:8]):  # plot first 8 vehicles
    arr, dep = sess['arrival_hour'], sess['departure_hour']
    soc = sess['soc_arrival']
    cap = sess['battery_capacity_kwh']
    soc_trace = []
    for h in range(24):
        if h < arr or h >= dep:
            soc_trace.append(None)
            continue
        kw = float(schedule[i, h]) if i < len(schedule) else 0
        if kw > 0:
            soc = min(soc + kw * 0.92 / cap, 0.95)
        elif kw < 0:
            soc = max(soc - (-kw) / 0.92 / cap, 0.10)
        soc_trace.append(soc)
    valid_h = [h for h, s in enumerate(soc_trace) if s is not None]
    valid_s = [s for s in soc_trace if s is not None]
    color = '#e65100' if sess['max_discharge_kw'] > 0 else '#1565c0'
    ax.plot(valid_h, valid_s, alpha=0.7, lw=1.5, color=color)
ax.axhline(0.80, color='red', lw=1, ls='--', alpha=0.7, label='80% departure min')
ax.axhline(0.95, color='gray', lw=1, ls=':', alpha=0.7, label='95% max SoC')
ax.set_ylabel('SoC')
ax.set_xlabel('Hour of Day')
ax.set_title('Per-Vehicle SoC Trajectory (blue=charge-only, orange=V2G capable)', fontsize=11)
ax.legend(fontsize=9)
ax.set_xticks(range(0, 24, 2))
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('strategy_deepdive.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compute and print economics for this example day
import importlib.util as ilu
spec2 = ilu.spec_from_file_location('prepare_v2g', 'prepare_v2g.py')
harness = ilu.module_from_spec(spec2)
spec2.loader.exec_module(harness)

result = harness.simulate_day(today_sessions, schedule, actual_prices)
print(f'Example day economics ({example_date}):')
print(f'  Gross revenue:      {result["gross_revenue_eur"]:+.2f} EUR')
print(f'  Degradation cost:   {result["degradation_cost_eur"]:-.2f} EUR')
print(f'  Net revenue:        {result["net_revenue_eur"]:+.2f} EUR')
print(f'  Charged:            {result["total_charged_kwh"]:.1f} kWh')
print(f'  Discharged (V2G):   {result["total_discharged_kwh"]:.1f} kWh')
print(f'  Violations:         {result["violation_count"]}')